# A tour of Marduk

This notebook walks through Marduk by example: PLAN's three opcodes, the BPLAN named ops, and a small program. Run each cell with **Shift+Enter**.

If you opened this notebook in a kernel other than **Marduk (PLAN)**, switch via *Kernel → Change Kernel*. If Marduk doesn't appear, install it with `python -m marduk install` and restart Jupyter.

## 1 — PLAN's three opcodes

PLAN has four constructors (Pin, Law, App, Nat) and exactly three opcodes:

| Opcode | Name | Arity | Purpose                                                  |
|--------|------|-------|----------------------------------------------------------|
| 0      | Pin  | 1     | Wrap a value as a content-addressed reference.           |
| 1      | Law  | 3     | Construct a named, fixed-arity pure function.            |
| 2      | Elim | 6     | Universal dispatch on Pin / Law / App / Nat constructor. |

Everything else is built on top, including arithmetic — those are the BPLAN named ops, which you'll meet in section 2.

### Pin — opcode 0

`#pin` constructs a Pin around a value. The default renderer collapses a Pin of a Law to `<{name…}>`; bare Pins of nats render as `<n>`.

In [ ]:
(#pin 5)

### Law — opcode 1

`#law` constructs a named function. The signature `(self arg1 arg2 …)` introduces local names: `self` is the recursive self-reference, the rest are arguments. The body is the last form; any earlier forms are let-binds.

An identity function:

In [ ]:
(#bind id
  (#pin
    (#law "id" (id x)
      x)))

(id 42)

### Elim — opcode 2

`Elim` takes six callbacks plus a scrutinee:

```
(Elim p l a z m o)
      │ │ │ │ │ │
      │ │ │ │ │ └── scrutinee
      │ │ │ │ └──── m: callback for Nat n+1, called as (m n)
      │ │ │ └────── z: value for Nat 0
      │ │ └──────── a: callback for App, called as (a fun arg)
      │ └────────── l: callback for Law, called as (l arity name body)
      └──────────── p: callback for Pin, called as (p inner)
```

A trivial use — `is_zero n`:

In [ ]:
(#bind is_zero
  (#pin
    (#law "is_zero" (is_zero n)
      (Elim 0 0 0 1 (#law "step" (step _) 0) n))))

(is_zero 0)

## 2 — BPLAN named ops

Arithmetic and a handful of inspection ops aren't part of PLAN proper; they live in BPLAN, dispatched at runtime by name. Marduk auto-binds wrappers for every BPLAN op on kernel start, so they work in cell 1.

Try:

In [ ]:
(Add 2 3)

In [ ]:
(Add 1 (Mul 2 3))

In [ ]:
(Inc 41)

Use `%env` to see what's bound:

In [ ]:
%env

(`%env` filters the BPLAN op prelude — those ~30 entries would otherwise drown user bindings.)

## 3 — A small program: Church booleans

Encode `True` and `False` as two-arg functions that pick their first or second argument:

In [ ]:
(#bind T (#pin (#law "T" (T a b) a)))
(#bind F (#pin (#law "F" (F a b) b)))

(#bind And
  (#pin
    (#law "And" (And p q)
      (p q F))))

In [ ]:
; (And T T 10 20) = T 10 20 = 10
; (And T F 10 20) = F 10 20 = 20
(And T F 10 20)

## Where to go next

- The fixtures in `tests/fixtures/` exercise S, Elim, and the rest in slightly more depth.
- The README's *Magic reference* lists every cell magic.
- The Reaver source ([`vendor/reaver/src/plan/`](https://github.com/xocore-tech/PLAN/tree/main/src/plan)) has substantial Plan Asm programs (`bst.plan`, `pretty.plan`, `stdlib.plan`) that this kernel can run cell-by-cell.